# Person 2 — RQ3 and RQ4: Transportation research analysis

This analysis implements the team's assigned research questions using the checked-in integrated snapshot. It was added during integration repair; it is not the missing original transportation EDA/cleaning notebook.

Sources: [Housing hg8x-zxpr](https://data.cityofnewyork.us/d/hg8x-zxpr) and [SIRTA xeqp-qz8h](https://data.cityofnewyork.us/d/xeqp-qz8h).
Run from the repository root. No live download is required.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(Path("app").resolve()))
from charts.rq3_rq4 import rq3, rq4

df_integrated = pd.read_csv("df_integrated.csv", dtype={"zip_code": "string"})
df_transport_clean = pd.read_csv("df_transport_clean.csv")
assert not df_integrated.duplicated(["zip_code", "year"]).any()
assert not df_integrated.isna().any().any()
print("Integrated shape:", df_integrated.shape)
print("Cleaned transportation snapshot dates:", df_transport_clean.crash_date.min(), df_transport_clean.crash_date.max())
display(df_integrated.groupby("year")[["affordable_units", "serious_crash_count", "severe_injuries"]].sum())


## RQ3 — How have affordable housing production and serious crashes changed over time?
Annual totals use two labelled axes because units and crash counts have different scales. Compare 2023–2025 for complete calendar years; 2022 and 2026 cover partial years. Matched ZIP coverage also changes by year, so this is not a citywide trend or a controlled comparison.


In [ ]:
rq3(df_integrated).show()
annual = df_integrated.groupby("year")[["affordable_units", "serious_crash_count"]].sum()
full_years = annual.loc[2023:2025]
display(full_years)
display(full_years.pct_change().mul(100).rename(columns=lambda c: c + "_change_percent"))


Interpret the displayed totals and percentage changes together: the series can rise or fall at different rates. Two lines moving together do not demonstrate that housing causes crashes. These totals are restricted to ZIP-years represented in both datasets.

## RQ4 — Do ZIP codes with more housing projects also have more severe crash injuries?
One point per ZIP, summed across selected years. Report Pearson and rank correlations as descriptive associations. Annual project counts can count a project in more than one ZIP/year; they are not unique citywide projects. Repeat on 2023–2025 to inspect sensitivity to partial years.


In [ ]:
rq4(df_integrated).show()
for label, data in [("All available years", df_integrated),
                    ("2023–2025", df_integrated[df_integrated.year.between(2023, 2025)])]:
    totals = data.groupby("zip_code")[["housing_project_count", "severe_injuries"]].sum()
    pearson = totals.housing_project_count.corr(totals.severe_injuries)
    spearman = totals.housing_project_count.rank().corr(totals.severe_injuries.rank())
    print(f"{label}: {len(totals)} ZIPs; Pearson r={pearson:.3f}; Spearman rho={spearman:.3f}")
    display(totals.nlargest(5, "severe_injuries"))


## Limits and teammate handoff
Population, road length, traffic volume and reporting coverage can influence both measures. Housing includes preservation, so these results do not isolate newly built housing. This notebook does not establish causal effects or per-person crash risk.

Still required from the transportation contributor: source loading code for xeqp-qz8h, raw snapshot or download/filter details, EDA, missing-value and outlier decisions, deduplication policy, and derivation of zip_code, borough and year. Do not infer those decisions from a cleaned CSV alone.
